In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from PIL import Image
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

print("Uploaded image:", file_name)

img = Image.open(file_name).convert("RGB")
img = img.resize((64, 64), Image.Resampling.LANCZOS)

image = np.array(img).astype(np.float32) / 255.0

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title("Original Color Image")
plt.axis("off")
plt.show()

X = image.reshape(1, 12288)

print("Original image size:", X.shape[1], "values")

input_size = 12288
hidden1_size = 512
hidden2_size = 256
compressed_size = 256

input_layer = Input(shape=(input_size,))

encoder = Dense(hidden1_size, activation="relu")(input_layer)
encoder = Dense(hidden2_size, activation="relu")(encoder)
compressed = Dense(compressed_size, activation="sigmoid")(encoder)

decoder = Dense(hidden2_size, activation="relu")(compressed)
decoder = Dense(hidden1_size, activation="relu")(decoder)
output_layer = Dense(input_size, activation="sigmoid")(decoder)

autoencoder = Model(input_layer, output_layer)
encoder_model = Model(input_layer, compressed)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

history = autoencoder.fit(
    X, X,
    epochs=100,
    batch_size=1,
    verbose=0
)

for epoch in range(20, 101, 20):
    print("Epoch:", epoch, "Loss:", history.history["loss"][epoch - 1])

plt.figure(figsize=(8, 5))
plt.plot(range(1, 101), history.history["loss"], linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Loss")
plt.yscale("log")
plt.grid(True)
plt.show()

compressed = encoder_model.predict(X, verbose=0)

print("\n==========================================")
print("COMPRESSED REPRESENTATION")
print("==========================================")

print("Original size   :", input_size, "values")
print("Compressed size :", compressed_size, "values")
print("\nCompressed values:")
print(compressed[0])

reconstructed = autoencoder.predict(X, verbose=0)
reconstructed_image = reconstructed.reshape(64, 64, 3)

compression_ratio = input_size / compressed_size
mse = np.mean((image - reconstructed_image) ** 2)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Original Color Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(compressed.reshape(16, 16), cmap="viridis")
plt.title("Compressed Representation\n256 Values")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(np.clip(reconstructed_image, 0, 1))
plt.title("Reconstructed Color Image")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\n==========================================")
print("FINAL RESULT")
print("==========================================")

print("Original image      : 64 x 64 x 3")
print("Original values     :", input_size)
print("Compressed values   :", compressed_size)
print("Compression Ratio   :", round(compression_ratio, 2), ": 1")
print("Reconstruction MSE  :", "{:.8f}".format(float(mse)))

print("\nImage compression completed successfully!")